This script combines data extraction from the SumUp dataset with pre-processing steps to prepare the data for exploration and model training.

In [ ]:
# For opening, viewing, and printing dataset info
import numpy as np
import pandas as pd

# For plotting data and metadata
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
matplotlib_inline.backend_inline.set_matplotlib_formats('png')

The data is split between Antarctica and Greenland. For the rest of this iteration analysis, our analyses will now focus on Antarctica

In [ ]:
SummedUp_raw = pd.read_csv("SUMup_2024_density_antarctica.csv")

In [ ]:
SummedUp_raw['density'] = SummedUp_raw['Density']/1000

In [ ]:
SummedUp_raw.to_csv('SummedUp_antdensity.csv')

HERE WE USE MATLAB TO INTERPOLATE TEMPERATURE AND SMB DATA FROM RACMO DATA FOR THE SUMMEDUP DATA

In [ ]:
'''
%import the datasets
tempdf = readtable('tskin.csv');
densitydf = readtable('SummedUp_antdensity.csv');
smbdf = readtable('smbdf.csv');

%extract important columns for temperature interpolation
templat = tempdf.lats;
templon = tempdf.lons;
temperature = tempdf.tskin;
smblat = smbdf.lats;
smblon = smbdf.lons;
smb = smbdf.smb;
denslat = densitydf.Latitude;
denslon = densitydf.Longitude;
density = densitydf.density;

F = scatteredInterpolant(templat,templon,temperature);
G = scatteredInterpolant(smblat,smblon,smb);
H = scatteredInterpolant(denslat,denslon,density);


denstemp = F(denslat,denslon);
denssmb = G(denslat,denslon);

smbtemp = F(smblat,smblon);
smbdensity = H(smblat,smblon);
%{
%density
plot3(denslat,denslon,density,'.',accumlat,accumlon,accumdens,'.'), grid on
title('Linear Interpolation')
xlabel('x'), ylabel('y'), zlabel('Values')
legend('Sample data','Interpolated query data','Location','Best')
%}

densitydf.Temperature = denstemp;
densitydf.smb = denssmb;
writetable(densitydf,'denssitynew2.csv')

smbdf.Temperature = smbtemp;
smbdf.Density = smbdensity;
writetable(smbdf,'smbnew_.csv')

'''

In [ ]:
Summednewdf = pd.read_csv('denssitynew2.csv')

In [ ]:
Summednewdf = Summednewdf.rename(columns={'smb':'Accumulation rate'})
Summednewdf = Summednewdf.rename(columns={'Midpoint':'Depth'})

In [ ]:
Summednewdf = Summednewdf.drop(columns = ['Var1','Unnamed_0','Error','SDOS_Flag','Density'])

In [ ]:
Summednewdf.to_csv('training_df.csv')

Now we explore the Antarctic Data

In [ ]:
SummedUp_df = SummedUp_df.drop(columns = ['Unnamed: 0','Profile','Citation','Method','Date','Timestamp','Start_Depth','Stop_Depth'])

In [ ]:
sns.set_theme(style="white")

# Generate a large random dataset
rs = np.random.RandomState(33)

# Compute the correlation matrix
corr = SummedUp_df.corr()

# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap 
sns.heatmap(corr, mask=mask, cmap=cmap,vmin=-1, vmax=1, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5});

In [ ]:
# Increase the size of the heatmap.
plt.figure(figsize=(16, 6))
heatmap = sns.heatmap(SummedUp_df.corr(), vmin=-1, vmax=1, annot=True)
heatmap.set_title('Correlation Heatmap', fontdict={'fontsize':12}, pad=12);

## Here we split the data

In [ ]:
df_all = pd.read_csv("training_df.csv",index_col=['Profile'])

First, we extract the profiles identified for visual testing 

In [ ]:
df_all3 = df_all.drop([2,24,26,1937,659,10,740])

In [ ]:
core_test = np.array([2,24,26,1937,659,10,740])
core_train = np.unique(df_all3.index.get_level_values(0).values)
df_train_all = df_all.loc[core_train]
df_test = df_all.loc[core_test]
df_train_all.shape, df_test.shape

## Extract the Validation dataset

In [ ]:
core_id_all = np.unique(df_train_all.index.get_level_values(0).values)

train_dataset, validation_dataset = train_test_split(core_id_all, 
                                               train_size=0.8,
                                               test_size=0.2, random_state = 917)

df_train = df_all.loc[train_dataset]
df_validation = df_all.loc[validation_dataset]
df_train.shape, df_validation.shape

In [ ]:
df_train.to_csv('df_train.csv')
df_validation.to_csv('df_validation.csv')
df_test.to_csv('df_test.csv')